In [33]:
import os
import polars as pl
import matplotlib.pyplot as plt

BASE_DIR = os.getcwd()
GPS_SRC = "/home/jovyan/DataTelkomsel"
GPS_DST = os.path.join(BASE_DIR, "processed-data")
os.makedirs(GPS_DST, exist_ok=True)

OUTPUT = os.path.join(GPS_DST, "all_gps_data_no_dup.parquet")

GPS_FILES = (
    ["2021Oktober/Oktober2021.csv"]
    + [f"2021November/November2021_part{i}.csv" for i in range(1, 8)]
    + [f"2021Desember/Desember2021_part{i}.csv" for i in range(1, 4)]
    + [f"2022Januari/Januari2022_part{i}.csv" for i in range(1, 3)]
    + [f"2022Februari/Februari2022_part{i}.csv" for i in range(1, 3)]
    + [f"2022Maret/Maret2022_part{i}.csv" for i in range(1, 3)]
    + ["2022April/April2022.csv"]
    + [f"2022Mei/Mei2022_part{i}.csv" for i in range(1, 3)]
    + ["2022Juni/Juni2022.csv"]
)

KEYS = ["maid", "latitude", "longitude", "timestamp"]

paths = [os.path.join(GPS_SRC, f) for f in GPS_FILES]
paths = [p for p in paths if os.path.exists(p)]

In [16]:
df = (
    pl.scan_csv(
        paths,
        schema_overrides={k: pl.Utf8 for k in KEYS},
    )
    .select(KEYS)
    .with_columns(
        pl.col("latitude").cast(pl.Float64, strict=False),
        pl.col("longitude").cast(pl.Float64, strict=False),
        pl.col("timestamp").cast(pl.Int64, strict=False),
    )
    .collect()
)

n_raw = df.height
df = df.drop_nulls(subset=KEYS).rechunk()
n_after_null = df.height
df = df.unique(subset=KEYS).sort(["maid", "timestamp"])
n_final = df.height

df.write_parquet(OUTPUT, compression="zstd")

print(f"Baris awal         : {n_raw:,}")
print(f"Dropped (NaN+hdr)  : {n_raw - n_after_null:,}")
print(f"Dropped duplicate  : {n_after_null - n_final:,}")
print(f"Baris final        : {n_final:,}")
print(f"Output             : {OUTPUT}")

Baris awal         : 321,969,096
Dropped (NaN+hdr)  : 237
Dropped duplicate  : 143,290,843
Baris final        : 178,678,016
Output             : /home/jovyan/Skripsi_Thesis_Disertasi/Farrel/processed-data/all_gps_data_no_dup.parquet


In [32]:
PARQUET = os.path.join("processed-data", "all_gps_data_no_dup.parquet")
df = pl.read_parquet(PARQUET)

n_rows, n_cols = df.shape

print("=" * 60)
print("DIMENSI DATASET")
print("=" * 60)
print(f"Rows         : {n_rows:,}")
print(f"Columns      : {n_cols}")
print(f"Total cells  : {n_rows * n_cols:,}")
print(f"Kolom        : {df.columns}")
print(f"Dtypes       : {dict(df.schema)}")

print("=" * 60)
print("UNIQUE ANALYSIS PER KOLOM")
print("=" * 60)
print(f"{'Kolom':<12} {'Unique':>15} {'Ratio':>10}")
print("-" * 40)
for col in df.columns:
    n_unique = df[col].n_unique()
    print(f"{col:<12} {n_unique:>15,} {n_unique / n_rows:>9.4%}")

print("=" * 60)
print("DUPLICATE ROWS ANALYSIS")
print("=" * 60)
n_distinct = df.unique().height
n_dups = n_rows - n_distinct
print(f"Total rows     : {n_rows:,}")
print(f"Distinct rows  : {n_distinct:,}")
print(f"Duplicate rows : {n_dups:,} ({n_dups / n_rows:.4%})")

print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)
print(f"{'Kolom':<12} {'Null':>12} {'Ratio':>10}")
print("-" * 36)
for col, n_null in df.null_count().row(0, named=True).items():
    print(f"{col:<12} {n_null:>12,} {n_null / n_rows:>9.4%}")

df = df.with_columns(ts_dt=pl.from_epoch("timestamp"))

print("=" * 60)
# ---------- Ringkasan umum ----------
print(f"File size        : {os.path.getsize(PARQUET) / 1e6:,.0f} MB")
print(f"Rows             : {df.height:,}")
print(f"Unique maid      : {df['maid'].n_unique():,}")
print(f"Schema           : {dict(df.schema)}")

# ---------- Temporal ----------
print(f"\nTemporal range   : {df['ts_dt'].min()} → {df['ts_dt'].max()}")

# ---------- Spatial ----------
lat_lo, lat_hi = df["latitude"].min(), df["latitude"].max()
lon_lo, lon_hi = df["longitude"].min(), df["longitude"].max()
print(f"Latitude  range  : [{lat_lo:.4f}, {lat_hi:.4f}]")
print(f"Longitude range  : [{lon_lo:.4f}, {lon_hi:.4f}]")

DIMENSI DATASET
Rows         : 178,678,016
Columns      : 4
Total cells  : 714,712,064
Kolom        : ['maid', 'latitude', 'longitude', 'timestamp']
Dtypes       : {'maid': String, 'latitude': Float64, 'longitude': Float64, 'timestamp': Int64}
UNIQUE ANALYSIS PER KOLOM
Kolom                 Unique      Ratio
----------------------------------------
maid               4,496,530   2.5166%
latitude             881,884   0.4936%
longitude            196,149   0.1098%
timestamp         17,484,089   9.7852%
DUPLICATE ROWS ANALYSIS
Total rows     : 178,678,016
Distinct rows  : 178,678,016
Duplicate rows : 0 (0.0000%)
MISSING VALUE ANALYSIS
Kolom                Null      Ratio
------------------------------------
maid                    0   0.0000%
latitude                0   0.0000%
longitude               0   0.0000%
timestamp               0   0.0000%
File size        : 1,294 MB
Rows             : 178,678,016
Unique maid      : 4,496,530
Schema           : {'maid': String, 'latitude': Float

In [ ]:
# ---------- Per-user ----------
users = df.group_by("maid").agg(
    n_pings=pl.len(),
    span_days=(pl.col("timestamp").max() - pl.col("timestamp").min()) / 86400,
)
print("\n--- Distribusi per-user ---")
print(users.select(["n_pings", "span_days"]).describe())
print(f"Users < 10 ping   : {users.filter(pl.col('n_pings') < 10).height:,}")
print(f"Users >= 1000 ping: {users.filter(pl.col('n_pings') >= 1000).height:,}")


--- Distribusi per-user ---
shape: (9, 3)
┌────────────┬────────────┬────────────┐
│ statistic  ┆ n_pings    ┆ span_days  │
│ ---        ┆ ---        ┆ ---        │
│ str        ┆ f64        ┆ f64        │
╞════════════╪════════════╪════════════╡
│ count      ┆ 4.49653e6  ┆ 4.49653e6  │
│ null_count ┆ 0.0        ┆ 0.0        │
│ mean       ┆ 39.736867  ┆ 31.04861   │
│ std        ┆ 515.244597 ┆ 56.75119   │
│ min        ┆ 1.0        ┆ 0.0        │
│ 25%        ┆ 1.0        ┆ 0.0        │
│ 50%        ┆ 3.0        ┆ 0.4171875  │
│ 75%        ┆ 12.0       ┆ 36.248495  │
│ max        ┆ 365780.0   ┆ 227.624248 │
└────────────┴────────────┴────────────┘
Users < 10 ping   : 3,216,670
Users >= 1000 ping: 27,634


In [ ]:
# ---------- Inter-ping delta (sampling rate) ----------
deltas = (
    df.select(delta_s=pl.col("timestamp").diff().over("maid"))
    .drop_nulls()
    .filter(pl.col("delta_s") > 0)
)
print("\n--- Inter-ping delta (detik) ---")
print(deltas.describe())


--- Inter-ping delta (detik) ---
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ delta_s       │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 1.18714063e8  │
│ null_count ┆ 0.0           │
│ mean       ┆ 101608.778821 │
│ std        ┆ 775658.502391 │
│ min        ┆ 1.0           │
│ 25%        ┆ 39.0          │
│ 50%        ┆ 147.0         │
│ 75%        ┆ 2482.0        │
│ max        ┆ 1.947378e7    │
└────────────┴───────────────┘
